In [2]:
import requests
import json
import re
import pandas as pd
import numpy as np
import sys

In [3]:
#define http_function to get the data from Uniprot API
def get_uniprot(accession: str) -> requests.models.Response:
    headers = {
      "accept": "application/json"
    }
    URL = f"https://rest.uniprot.org/uniprotkb/{accession}"

    return requests.get(URL, headers=headers)


get_uniprot('P11473')
#get_uniprot('meow')
#get_uniprot('helloworld').json()

<Response [200]>

In [13]:
def uniprot_parse_response(resp: requests.models.Response):
    if not resp.ok:
        resp.raise_for_status()
        sys.exit()

    resp_json = resp.json()
    organism = resp_json['organism']['scientificName']
    geneInfo = resp_json['genes']
    sequence = resp_json['sequence']
  
    output = f"ORGANISM(scientific name): {organism}; GENE INFO: {geneInfo}; SEQUENCE: {sequence}"
    return output


uniprot_parse_response(get_uniprot('P11473'))

"ORGANISM(scientific name): Homo sapiens; GENE INFO: [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]; SEQUENCE: {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS', 'length': 427, 'molWeight': 48289, 'crc64': 'F95F300D042C4CB7', 'md5': '0D963ACD4A34674368324EE026023597'}"

In [16]:
def get_ensembl(id):
    headers = {
      "accept": "application/json"
    }
    url = f"https://rest.ensembl.org/lookup/id/{id}"

    return requests.get(url, headers=headers)


get_ensembl('ENSMUSG00000041147').json()

{'logic_name': 'ensembl_havana_gene_mus_musculus',
 'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
 'display_name': 'Brca2',
 'seq_region_name': '5',
 'object_type': 'Gene',
 'source': 'ensembl_havana',
 'db_type': 'core',
 'biotype': 'protein_coding',
 'assembly_name': 'GRCm39',
 'start': 150446095,
 'species': 'mus_musculus',
 'canonical_transcript': 'ENSMUST00000044620.11',
 'id': 'ENSMUSG00000041147',
 'end': 150493794,
 'version': 11,
 'strand': 1}

In [17]:
 # parse Ensembl response and output
 # object_type, assembly_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source
 # do not forget to include error handling (e.g. for key errors)
def ensembl_parse_response(resp: requests.models.Response):
    if not resp.ok:
        resp.raise_for_status()
        sys.exit()

    resp_json= resp.json()
    object_type = resp_json['object_type']
    ass_name = resp_json['assembly_name']
    species = resp_json['species']
    db_type = resp_json['db_type']
    biotype = resp_json['biotype']
    display_name = resp_json['display_name']
    id = resp_json['id']
    description = resp_json['description']
    canonical_transcript = resp_json['canonical_transcript']
    source = resp_json['source']
    
    data = [object_type, ass_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source]

    return data


ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))


['Gene',
 'GRCm39',
 'mus_musculus',
 'core',
 'protein_coding',
 'Brca2',
 'ENSMUSG00000041147',
 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
 'ENSMUST00000044620.11',
 'ensembl_havana']

In [31]:
def main(ids: list) -> dict:
    uni_reg = re.compile(r'^[A-Z][0-9][A-Z0-9]{4}$')
    ens_reg = re.compile(r'^ENS[A-Z]*[0-9]+(\.[0-9]+)?$')
    output = {}

    for id in ids:
        try:
            if uni_reg.match(id):
                resp = get_uniprot(id)
                info = uniprot_parse_response(resp)
                output[id] = info
            elif ens_reg.match(id):
                resp = get_ensembl(id)
                info = ensembl_parse_response(resp)
                output[id] = info
            else:
                output[id] = "error:unknown database"

        except requests.exceptions.RequestException as e:
            output[id] = f"request failed: {e}"
        except Exception as e:
            output[id] = f"error: {e}"

    return output


main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])


{'P11473': "ORGANISM(scientific name): Homo sapiens; GENE INFO: [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]; SEQUENCE: {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS', 'length': 427, 'molWeight': 48289, 'crc64': 'F95F300D042C4CB7', 'md5': '0D963ACD4A34674368324EE026023597'}",
 'Q91XI3': "ORGANISM(scientific name): Ictidomys tridecemlineatus; GENE INFO: [{'geneName': {'value': 'INS'}}]; SEQUENCE: {'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHLVEALYLVCGERGFFYTPKSRREVEEQQGGQVELGGGPGAGLPQPLALEMALQKR